# DSA 8301 — Kenya Housing Survey 2023/24
## Statistical Analysis · Parametric & Nonparametric Inference on the HFVS

**Student:** Sephine Valerie Jerono | **No:** 222331
**Supervisor:** Dr. John Olukuru
**Institution:** Strathmore Institute of Mathematical Sciences (iLabAfrica)
**Input:** `model_ready.csv` — output of the KHS_Clean_Pipeline notebook (21,347 households × 64 columns)

---

## Pipeline Architecture

This notebook continues the disciplined pipeline structure of the cleaning notebook. Seven sequential
stages (**SA-00 → SA-06**), each producing a documented, evidence-based output that the next stage
consumes. **No statistical test is selected by default — every parametric/nonparametric choice is
routed by evidence produced upstream in this same notebook.**

| Pipeline | Name | Input | Output |
|---|---|---|---|
| **SA-00** | Data Contract & Quality Gate | `model_ready.csv` | validated `df`, quality flags, fingerprint check |
| **SA-01** | Descriptive Profiling | `df` | weighted/unweighted stats, distribution diagnostics, `normality_df` |
| **SA-02** | Assumption-Driven Method Router | `normality_df` + group sizes | `test_plan` — which method, on which variable, and why |
| **SA-03** | Parametric Inference Battery | `df` + `test_plan` | CIs, t-tests, ANOVA, regression — only where licensed |
| **SA-04** | Nonparametric Inference Battery | `df` + `test_plan` | rank tests, Spearman, bootstrap — for skewed/ordinal/small-group cases |
| **SA-05** | Cross-Validation & Synthesis | SA-03 + SA-04 outputs | side-by-side comparison, effect sizes, agreement check |
| **SA-06** | Findings Summary & Export | all prior | `sa_findings_summary.csv`, final interpretive narrative |

### Data-quality carry-overs from the cleaning pipeline (handled explicitly below)

> **`yrs_in_dwelling` — EXCLUDED.** 70.7% of rows (15,088 of 21,347) hold a raw calendar move-in year
> (1940–2024) instead of a duration; the remaining 29.3% are all exactly `1.0` (a default/fallback,
> not genuine variation). This column was 1 of only 7 surviving inputs to `hfvs_d2_tenure`
> (~14.3% of D2's weight → ~2.9% direct weight in `hfvs_composite`). Excluded from all SA analyses;
> impact on D2/composite is bounded and noted but not corrected (requires the true survey interview
> year per household, not recoverable from this file).
>
> **`mean_age` — 3 rows excluded from `mean_age`-specific analyses only.** Two rows contain impossible
> negative ages (-2488.25, -1986.20); one contains a stray calendar year (2024.0) instead of an age.
> All other columns for these 3 households are retained.
>
> **`hh_weight` (survey design weight) — used throughout.** Ranges 24.9–8,162.0 (≈328× spread).
> Per design decision: every descriptive statistic in SA-01 is reported **both unweighted and
> weighted**, with divergence >5% flagged explicitly.
>
> **Vulnerability grouping — two definitions carried in parallel.** The dissertation's official
> `high_vulnerability` (`hfvs_composite > 0.50`) yields only **85 positive cases** (0.40%) — too few
> for well-powered group comparisons. An alternative **top-quartile** grouping
> (`hfvs_composite ≥ Q75 = 0.359`) yields **5,337 positive cases** and is *fully nested* inside the
> official definition (every official-positive household is also top-quartile-positive). Both are
> carried side-by-side through SA-03–SA-05 so the dissertation can report the formal threshold while
> having an adequately powered comparison group available.

---

## SA-00 — Data Contract, Quality Gate & Environment

Before any statistic is computed, we re-verify that the file handed to this notebook is *exactly*
the dataset the cleaning pipeline produced — not a stale copy, not a re-export with drift. We do this
by checking shape, integrity, and a numeric **fingerprint** against the cleaning notebook's own
printed output (PL-07.6). If any of these checks fail, every downstream result in this notebook would
be untrustworthy, so we `assert` rather than warn.

In [24]:
# ── PL-00.1  Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

# ── PL-00.4  Paths ────────────────────────────────────────────────────────
DRIVE  = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ     = DRIVE / 'data' / 'parquet'
FIGS   = DRIVE / 'outputs' / 'figures'
TABS   = DRIVE / 'outputs' / 'tables'

for p in [FIGS, TABS]:
    p.mkdir(parents=True, exist_ok=True)

MASTER_PATH = TABS / 'model_ready.csv'
print(f'model_ready.csv exists: {MASTER_PATH.exists()}')





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
model_ready.csv exists: True


In [25]:
# ── SA-00.1  Imports & shared configuration ──────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
RNG = np.random.RandomState(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 70)
pd.set_option('display.max_rows', 120)

# Colour palette consistent with the cleaning notebook
RED, AMBER, BLUE, PURPLE, TEAL, DARK = '#D7263D','#F2A541','#2E86AB','#6B4E9B','#1B998B','#1A1A2E'
GREY = '#9AA5B1'

DATA = Path('.')
FIGS = Path('figs'); FIGS.mkdir(exist_ok=True)
TABS = Path('tabs'); TABS.mkdir(exist_ok=True)

print('Environment ready.')

Environment ready.


In [26]:
# ── SA-00.2  Load & shape contract ────────────────────────────────────────
df = pd.read_csv(MASTER_PATH)


print(f'Loaded master_frame: {df.shape[0]:,} rows × {df.shape[1]} columns')


EXPECTED_ROWS, EXPECTED_COLS = 21347, 64
assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), f'Shape contract violated: got {df.shape}'
print(f'Shape OK: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded master_frame: 21,347 rows × 64 columns
Shape OK: 21,347 rows x 64 columns


In [27]:
# ── SA-00.3  Integrity contract ───────────────────────────────────────────
n_dup     = df['hh_id'].duplicated().sum()
n_missing = df.isna().sum().sum()

print(f'Duplicate hh_id      : {n_dup}')
print(f'Missing cells (total): {n_missing}')

assert n_dup == 0,     'Unexpected duplicate household identifiers'
assert n_missing == 0, 'Unexpected missing values — cleaning pipeline should have resolved all'
print('Integrity contract satisfied.')

Duplicate hh_id      : 0
Missing cells (total): 0
Integrity contract satisfied.


In [28]:
# ── SA-00.4  Fingerprint validation against cleaning-notebook ground truth ─
# These reference values are the exact printed output of PL-07.6 in KHS_Clean_Pipeline.ipynb.
# If these don't match, this CSV is not the same data the cleaning notebook produced.
FINGERPRINT = {
    'hfvs_composite'        : (0.3218, 0.0605),
    'hfvs_d1_financial'     : (0.3712, 0.0816),
    'hfvs_d2_tenure'        : (0.3877, 0.1723),
    'hfvs_d3_hazard'        : (0.1210, 0.1802),
    'hfvs_d4_quality'       : (0.2554, 0.1515),
    'hfvs_d5_utility'       : (0.4693, 0.1948),
    'log_total_expenditure' : (10.3893, 0.6722),
}

print(f"{'column':<24}{'mean':>10}{'ref_mean':>10}{'std':>10}{'ref_std':>10}{'status':>10}")
all_match = True
for col, (m_ref, s_ref) in FINGERPRINT.items():
    m, s = df[col].mean(), df[col].std()
    ok = abs(m - m_ref) < 0.001 and abs(s - s_ref) < 0.001
    all_match &= ok
    print(f"{col:<24}{m:>10.4f}{m_ref:>10.4f}{s:>10.4f}{s_ref:>10.4f}{'OK' if ok else 'MISMATCH':>10}")

assert all_match, 'Fingerprint mismatch — this file does not match the cleaning pipeline output.'
print('\nFingerprint confirmed. Data lineage from KHS_Clean_Pipeline.ipynb verified.')

column                        mean  ref_mean       std   ref_std    status
hfvs_composite              0.3218    0.3218    0.0605    0.0605        OK
hfvs_d1_financial           0.3712    0.3712    0.0816    0.0816        OK
hfvs_d2_tenure              0.3877    0.3877    0.1723    0.1723        OK
hfvs_d3_hazard              0.1210    0.1210    0.1802    0.1802        OK
hfvs_d4_quality             0.2554    0.2554    0.1515    0.1515        OK
hfvs_d5_utility             0.4693    0.4693    0.1948    0.1948        OK
log_total_expenditure      10.3893   10.3893    0.6722    0.6722        OK

Fingerprint confirmed. Data lineage from KHS_Clean_Pipeline.ipynb verified.


In [29]:
# ── SA-00.5  Known data-quality casualty: yrs_in_dwelling (EXCLUDED) ──────
# Evidence (do not re-derive — this is a documented finding from notebook development):
yid = pd.to_numeric(df['yrs_in_dwelling'], errors='coerce') if 'yrs_in_dwelling' in df.columns else None
if yid is not None:
    n_yearlike = ((yid >= 1900) & (yid <= 2030)).sum()
    n_default1 = (yid == 1.0).sum()
    print('yrs_in_dwelling diagnostic (column retained in df, but EXCLUDED from all SA analyses below):')
    print(f'  Rows holding a raw calendar year (1900-2030): {n_yearlike:,} ({100*n_yearlike/len(df):.1f}%)')
    print(f'  Rows fixed at exactly 1.0 (suspected default) : {n_default1:,} ({100*n_default1/len(df):.1f}%)')
    print(f'  Combined unusable share                       : {100*(n_yearlike+n_default1)/len(df):.1f}%')
    print()
    print('  Decision: EXCLUDE entirely from this notebook. It contributed to hfvs_d2_tenure')
    print('  (1 of 7 surviving D2 inputs => ~14.3% of D2 weight => ~2.9% of hfvs_composite weight),')
    print('  but a correct repair requires the true per-household survey interview year, which is')
    print('  not present in this file. No analysis below references this column.')
EXCLUDED_COLUMNS = ['yrs_in_dwelling']

yrs_in_dwelling diagnostic (column retained in df, but EXCLUDED from all SA analyses below):
  Rows holding a raw calendar year (1900-2030): 15,088 (70.7%)
  Rows fixed at exactly 1.0 (suspected default) : 6,259 (29.3%)
  Combined unusable share                       : 100.0%

  Decision: EXCLUDE entirely from this notebook. It contributed to hfvs_d2_tenure
  (1 of 7 surviving D2 inputs => ~14.3% of D2 weight => ~2.9% of hfvs_composite weight),
  but a correct repair requires the true per-household survey interview year, which is
  not present in this file. No analysis below references this column.


In [30]:
# ── SA-00.6  Known data-quality casualty: mean_age (3 rows excluded) ─────
bad_age_mask = (df['mean_age'] < 0) | (df['mean_age'] > 115)
bad_age_ids  = df.loc[bad_age_mask, 'hh_id'].tolist()

print(f'Implausible mean_age rows: {bad_age_mask.sum()}')
print(df.loc[bad_age_mask, ['hh_id', 'mean_age', 'hh_size']].to_string(index=False))
print()
print('Decision: exclude these 3 rows from mean_age-specific analyses only.')
print('All other columns for these 3 households remain in every other analysis.')

MEAN_AGE_VALID = ~bad_age_mask  # boolean mask: True = usable for mean_age analyses

Implausible mean_age rows: 3
      hh_id   mean_age  hh_size
06-45-62-98 -2488.2500   5.0000
22-65-03-56  2024.0000   2.0000
66-72-63-06 -1986.2000   5.0000

Decision: exclude these 3 rows from mean_age-specific analyses only.
All other columns for these 3 households remain in every other analysis.


In [31]:
# ── SA-00.7  Survey design weight overview ────────────────────────────────
print(df['hh_weight'].describe())
print(f"\nWeight ratio (max/min): {df['hh_weight'].max()/df['hh_weight'].min():.1f}x")
print('A ~328x spread means unweighted statistics may not represent the national population')
print('correctly if sampling probability varies by stratum (urban/rural, county). SA-01 reports')
print('both unweighted and weighted statistics for every variable as a result.')

count   21347.0000
mean      650.4955
std       615.4876
min        24.9126
25%       239.4361
50%       549.3494
75%       845.8809
max      8162.0115
Name: hh_weight, dtype: float64

Weight ratio (max/min): 327.6x
A ~328x spread means unweighted statistics may not represent the national population
correctly if sampling probability varies by stratum (urban/rural, county). SA-01 reports
both unweighted and weighted statistics for every variable as a result.


## SA-01 — Descriptive Profiling

**Research question this stage answers:** *What does each variable actually look like, and is it
safe to summarise it with a single mean — or does the survey design / shape of the distribution
require something more careful?*

This stage produces the **evidence** that SA-02 will use to route every later test. We deliberately
compute descriptive statistics in **two parallel tracks** — unweighted (treats every household as a
single observation) and weighted (uses `hh_weight` to approximate the national population) — because
the ~328× weight spread found in SA-00 means these can diverge.

In [32]:
# ── SA-01.1  Variable inventory for descriptive profiling ────────────────
# yrs_in_dwelling is excluded (SA-00.5). All other genuinely continuous HFVS inputs + outputs included.
CONTINUOUS_VARS = [
    'log_total_expenditure', 'log_housing_cost', 'housing_burden_ratio', 'utility_burden_ratio',
    'log_rent', 'perception_quality_score', 'log_floor_area', 'dwelling_age_yrs',
    'hh_size', 'dependency_ratio', 'mean_age', 'cty_housing_gap_ratio', 'wsvc_sewer_conns',
    'hfvs_d1_financial', 'hfvs_d2_tenure', 'hfvs_d3_hazard', 'hfvs_d4_quality', 'hfvs_d5_utility',
    'hfvs_composite',
]
print(f'{len(CONTINUOUS_VARS)} continuous variables queued for descriptive profiling.')

19 continuous variables queued for descriptive profiling.


In [33]:
# ── SA-01.2  Weighted statistics helpers ──────────────────────────────────
def weighted_mean(x, w):
    return np.average(x, weights=w)

def weighted_std(x, w):
    m = weighted_mean(x, w)
    var = np.average((x - m) ** 2, weights=w)
    return np.sqrt(var)

def weighted_quantile(x, w, q):
    order = np.argsort(x)
    x_sorted, w_sorted = np.asarray(x)[order], np.asarray(w)[order]
    cum_w = (np.cumsum(w_sorted) - 0.5 * w_sorted) / w_sorted.sum()
    return np.interp(q, cum_w, x_sorted)

print('Weighted-statistics helpers defined.')

Weighted-statistics helpers defined.


In [34]:
# ── SA-01.3  Weighted vs unweighted descriptive table ─────────────────────
EXCLUDE_FROM = {'mean_age': ~MEAN_AGE_VALID}  # SA-00.6

records = []
for col in CONTINUOUS_VARS:
    mask = EXCLUDE_FROM.get(col)
    sub  = df.loc[~mask] if mask is not None else df
    x, w = sub[col].values, sub['hh_weight'].values

    mean_u, std_u = x.mean(), x.std()
    mean_w, std_w = weighted_mean(x, w), weighted_std(x, w)
    pct_diff = 100 * (mean_w - mean_u) / mean_u if mean_u != 0 else np.nan

    records.append({
        'variable': col, 'n': len(x),
        'mean_unweighted': mean_u, 'mean_weighted': mean_w, 'pct_diff': pct_diff,
        'median_unweighted': np.median(x), 'median_weighted': weighted_quantile(x, w, 0.5),
        'std_unweighted': std_u, 'std_weighted': std_w,
        'iqr_unweighted': np.percentile(x, 75) - np.percentile(x, 25),
        'range_unweighted': x.max() - x.min(),
        'skew': stats.skew(x), 'excess_kurtosis': stats.kurtosis(x),
        'excluded_rows': int(mask.sum()) if mask is not None else 0,
    })

desc_stats = pd.DataFrame(records).set_index('variable')
desc_stats['flag_weight_divergence'] = desc_stats['pct_diff'].abs() > 5

print(desc_stats[['n','mean_unweighted','mean_weighted','pct_diff','median_unweighted',
                   'std_unweighted','std_weighted','skew','excess_kurtosis']].round(4).to_string())

                              n  mean_unweighted  mean_weighted  pct_diff  median_unweighted  std_unweighted  std_weighted    skew  excess_kurtosis
variable                                                                                                                                           
log_total_expenditure     21347          10.3893        10.3992    0.0948            10.3735          0.6722        0.6463 -0.0578           0.4086
log_housing_cost          21347           6.4367         6.5343    1.5177             7.3139          3.1162        3.1480 -1.1316           0.2354
housing_burden_ratio      21347           0.0921         0.0987    7.1149             0.0539          0.1187        0.1174  2.5762           8.9664
utility_burden_ratio      21347           0.4106         0.3998   -2.6366             0.3921          0.2498        0.2472  0.2280          -1.0316
log_rent                  21347           2.9656         3.0620    3.2527             0.0000          3.8916    

In [35]:
# ── SA-01.4  Weight-divergence flag — where does survey weighting matter? ─
flagged = desc_stats.loc[desc_stats['flag_weight_divergence'],
                          ['mean_unweighted', 'mean_weighted', 'pct_diff']]
print('Variables where weighted and unweighted means diverge by >5%:')
print(flagged.round(4).to_string())
print()
print('Interpretation: hfvs_composite itself diverges only',
      f"{desc_stats.loc['hfvs_composite','pct_diff']:.2f}%", 'between weighted and unweighted —')
print('the headline HFVS construction is fairly weight-robust even though some of its raw inputs')
print('(e.g. dwelling_age_yrs, dependency_ratio) are not. wsvc_sewer_conns shows the largest')
print('divergence (56.7%) — this is a county-level infrastructure variable, so it is highly sensitive')
print('to which counties are over/under-weighted by the survey design; treat its unweighted mean')
print('with caution in any write-up.')

desc_stats.to_csv(TABS / 'sa01_descriptive_statistics.csv')
print('\nSaved sa01_descriptive_statistics.csv')

Variables where weighted and unweighted means diverge by >5%:
                      mean_unweighted  mean_weighted  pct_diff
variable                                                      
housing_burden_ratio           0.0921         0.0987    7.1149
dwelling_age_yrs              12.7134        13.4104    5.4822
dependency_ratio               0.7473         0.7059   -5.5413
wsvc_sewer_conns            1733.5893      2716.4092   56.6928
hfvs_d3_hazard                 0.1210         0.1273    5.1702

Interpretation: hfvs_composite itself diverges only -0.80% between weighted and unweighted —
the headline HFVS construction is fairly weight-robust even though some of its raw inputs
(e.g. dwelling_age_yrs, dependency_ratio) are not. wsvc_sewer_conns shows the largest
divergence (56.7%) — this is a county-level infrastructure variable, so it is highly sensitive
to which counties are over/under-weighted by the survey design; treat its unweighted mean
with caution in any write-up.

Saved sa01_

In [36]:
# ── SA-01.5  Normality diagnostics (sample-size-robust) ───────────────────
# Shapiro-Wilk on the FULL sample (n=21,347) will reject almost any real-world distribution purely
# from statistical power, not necessarily meaningful non-normality. We therefore:
#   1. Report Shapiro-Wilk on a FIXED random subsample (n=2000) for a fair, consistent power level.
#   2. Anchor the actual verdict on skewness / excess-kurtosis — sample-size-invariant effect sizes.
#   3. Flag any disagreement between the two criteria explicitly, rather than silently picking one.

def normality_verdict(x, n_total, random_state=42):
    x = np.asarray(x)
    skewness, kurt = stats.skew(x), stats.kurtosis(x)

    rng = np.random.RandomState(random_state)
    sub = rng.choice(x, size=min(2000, len(x)), replace=False)
    _, sw_p = stats.shapiro(sub)

    effect_size_normal = abs(skewness) < 0.5 and abs(kurt) < 1.0
    sw_normal = sw_p > 0.05

    if effect_size_normal and sw_normal:
        verdict = 'approximately normal'
    elif effect_size_normal and not sw_normal:
        verdict = 'approx. normal (SW rejects on power alone; effect sizes negligible)'
    elif not effect_size_normal and not sw_normal:
        verdict = 'non-normal'
    else:
        verdict = 'borderline — inspect Q-Q plot'

    return pd.Series({'n': n_total, 'skew': skewness, 'excess_kurtosis': kurt,
                       'shapiro_p_n2000': sw_p, 'effect_size_normal': effect_size_normal,
                       'verdict': verdict})

normality_rows = []
for col in CONTINUOUS_VARS:
    mask = EXCLUDE_FROM.get(col)
    x = df.loc[~mask, col] if mask is not None else df[col]
    normality_rows.append(normality_verdict(x.values, len(x)).rename(col))

normality_df = pd.DataFrame(normality_rows)
normality_df.index.name = 'variable'
pd.set_option('display.max_colwidth', 65)
print(normality_df.round(4).to_string())

                              n    skew  excess_kurtosis  shapiro_p_n2000  effect_size_normal                                                              verdict
variable                                                                                                                                                          
log_total_expenditure     21347 -0.0578           0.4086           0.0002                True  approx. normal (SW rejects on power alone; effect sizes negligible)
log_housing_cost          21347 -1.1316           0.2354           0.0000               False                                                           non-normal
housing_burden_ratio      21347  2.5762           8.9664           0.0000               False                                                           non-normal
utility_burden_ratio      21347  0.2280          -1.0316           0.0000               False                                                           non-normal
log_rent              

In [37]:
# ── SA-01.6  Normality verdict summary — this drives SA-02's routing ─────
print(normality_df['verdict'].value_counts().to_string())
normality_df.to_csv(TABS / 'sa01_normality_diagnostics.csv')
print('\nSaved sa01_normality_diagnostics.csv — SA-02 reads this directly.')

verdict
non-normal                                                             12
approx. normal (SW rejects on power alone; effect sizes negligible)     7

Saved sa01_normality_diagnostics.csv — SA-02 reads this directly.
